# Урок 13. Деревья и иерархические структуры

11 класс · II четверть

[⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [← Урок 12](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-12.ipynb) · [Урок 14 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-14.ipynb)

---

Дерево, корень, поддеревья, высота. Обходы дерева. Дерево решений и дерево игры. Иерархии в реальных данных.

In [ ]:
#@title 🚀 Регистрация { display-mode: "form" }
#@markdown Впиши свои данные и запусти ячейку (Shift+Enter).
#@markdown Если заводил секрет `INFORMATIKA` и включил к нему доступ
#@markdown (🔑 на панели слева) — поля можно оставить пустыми.
#@markdown
#@markdown Colab спросит разрешение на доступ к аккаунту — нажми
#@markdown «Разрешить». Так проверка понимает, чья это работа.
ФИО = "" #@param {type:"string"}
#@markdown Класс — с буквой, например 11А
Класс = "" #@param {type:"string"}
#@markdown
#@markdown Адрес журнала даёт учитель. Пусто — в конце урока
#@markdown получишь квитанцию, её нужно будет отправить ему сам.
Журнал = "" #@param {type:"string"}

import importlib, urllib.request
urllib.request.urlretrieve("https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/lib/schoolinf.py", "schoolinf.py")
import schoolinf as si
importlib.reload(si)
si.start(lesson="11-13", name=ФИО, klass=Класс,
         journal=Журнал)

## Разбираемся

### Дерево как способ навести порядок

<img src="https://raw.githubusercontent.com/LeksssGray/informatika-7-11/main/img/g11/drevo.jpg" width="260" alt="«Родословное древо человека», Эрнст Геккель, 1874 год">

*«Родословное древо человека», Эрнст Геккель, 1874 год*

<sub>Ernst Haeckel · Public domain · Wikimedia Commons</sub>

Биологи рисовали деревья задолго до появления компьютеров: на гравюре
Геккеля 1874 года всё живое разложено по ветвям от общего ствола.
Идея та же, что и в информатике: если у каждого объекта ровно один
«родитель», получается структура, в которой мгновенно понятно, что
к чему относится.

Так устроены файловая система, оглавление книги, структура сайта,
организация компании, разбор выражения компилятором и дерево ходов
в шахматной программе.

### Определение и словарь

**Дерево** — связный граф без циклов. Обычно у него выделен **корень**,
и тогда каждая вершина, кроме корня, имеет ровно одного родителя.

| Термин | Значение |
|---|---|
| корень | вершина без родителя, начало дерева |
| потомок (ребёнок) | вершина уровнем ниже |
| лист | вершина без потомков |
| внутренняя вершина | вершина, у которой есть потомки |
| уровень (глубина) | сколько рёбер от корня до вершины |
| высота | наибольший уровень в дереве |
| поддерево | вершина вместе со всеми её потомками |

Полезные факты: в дереве из n вершин ровно **n − 1** рёбер, и путь
между любыми двумя вершинами единственный.

### Как хранить дерево

Чаще всего — словарём «вершина → список потомков»:

```python
{"А": ["Б", "В"], "Б": ["Г", "Д"], "В": ["Е"], "Г": [], "Д": [], "Е": []}
```

Иногда удобнее наоборот, «вершина → родитель» — например, когда
нужно быстро подниматься к корню.

### Рекурсия — родной язык деревьев

Поддерево — это тоже дерево. Поэтому почти любая характеристика
считается одинаково:

```
что-то(вершина) = обработать саму вершину + то же самое для каждого потомка
```

Базовый случай приходит сам собой: у листа потомков нет, цикл `for`
не выполняется ни разу, и рекурсия останавливается.

### Три обхода

| Обход | Порядок | Где применяется |
|---|---|---|
| прямой (preorder) | вершина, потом поддеревья | вывод оглавления, копирование |
| обратный (postorder) | сначала поддеревья, потом вершина | подсчёт размера папки, удаление |
| в ширину | по уровням | поиск ближайшего, вывод по этажам |

Разница между прямым и обратным — одна строка кода, а смысл разный.
Размер папки можно узнать, только когда посчитаны все вложенные, —
это обратный обход. Напечатать оглавление нужно сверху вниз — прямой.

### Дерево игры

Особый случай — **дерево решений**: вершины это позиции, рёбра — ходы.
Полное дерево игры в крестики-нолики содержит около 250 тысяч
позиций, и его можно перебрать целиком. У шахмат вариантов больше,
чем атомов в наблюдаемой Вселенной, поэтому дерево обрезают:
считают на несколько ходов вперёд и оценивают позицию приближённо.

## Смотрим, как это работает

### Пример 1. Дерево файловой системы

In [ ]:
дерево = {
    "/": ["документы", "фото", "заметка.txt"],
    "документы": ["отчёт.docx", "смета.xlsx"],
    "фото": ["лето", "кот.jpg"],
    "лето": ["море.jpg", "горы.jpg"],
    "отчёт.docx": [], "смета.xlsx": [], "заметка.txt": [],
    "кот.jpg": [], "море.jpg": [], "горы.jpg": [],
}


def напечатать(дерево, вершина, уровень=0):
    print("   " * уровень + ("└─ " if уровень else "") + вершина)
    for потомок in дерево[вершина]:
        напечатать(дерево, потомок, уровень + 1)


напечатать(дерево, "/")

Это прямой обход: сначала печатаем вершину, потом спускаемся
в потомков. Уровень передаётся в рекурсию и превращается в отступ.

### Пример 2. Характеристики дерева

In [ ]:
def вершин(дерево, вершина):
    итог = 1
    for потомок in дерево[вершина]:
        итог += вершин(дерево, потомок)
    return итог


def листьев(дерево, вершина):
    if not дерево[вершина]:
        return 1
    return sum(листьев(дерево, потомок) for потомок in дерево[вершина])


def высота(дерево, вершина):
    if not дерево[вершина]:
        return 0
    return 1 + max(высота(дерево, потомок) for потомок in дерево[вершина])


print("Вершин:", вершин(дерево, "/"))
print("Листьев:", листьев(дерево, "/"))
print("Высота:", высота(дерево, "/"))
print("Рёбер:", вершин(дерево, "/") - 1)

Три функции устроены совершенно одинаково: обработали вершину,
сложили результаты потомков. Отличается только то, что именно
складываем и как обрабатываем лист.

### Пример 3. Обратный обход: размер папки

In [ ]:
размеры = {
    "заметка.txt": 2, "отчёт.docx": 40, "смета.xlsx": 25,
    "кот.jpg": 300, "море.jpg": 500, "горы.jpg": 450,
}


def размер(дерево, вершина):
    if not дерево[вершина]:
        return размеры[вершина]
    return sum(размер(дерево, потомок) for потомок in дерево[вершина])


for папка in ("лето", "фото", "документы", "/"):
    print(f"{папка:<11} {размер(дерево, папка):>5} КБ")

Размер папки нельзя узнать, не посчитав вложенные, — поэтому здесь
обратный обход: сначала потомки, потом вершина.

### Пример 4. Обход в ширину: по уровням

In [ ]:
def по_уровням(дерево, корень):
    уровни = {}
    очередь = [(корень, 0)]
    while очередь:
        вершина, уровень = очередь.pop(0)
        if уровень not in уровни:
            уровни[уровень] = []
        уровни[уровень].append(вершина)
        for потомок in дерево[вершина]:
            очередь.append((потомок, уровень + 1))
    return уровни


for уровень, вершины in по_уровням(дерево, "/").items():
    print(f"уровень {уровень}: {', '.join(вершины)}")

### Пример 5. Путь от корня до вершины

Единственность пути в дереве позволяет искать его простым спуском:
как только нашли — возвращаем.

In [ ]:
def путь(дерево, вершина, цель):
    if вершина == цель:
        return [вершина]
    for потомок in дерево[вершина]:
        найденный = путь(дерево, потомок, цель)
        if найденный:
            return [вершина] + найденный
    return None


print(" / ".join(путь(дерево, "/", "горы.jpg")))
print(путь(дерево, "/", "нет.txt"))

Возврат `None` из тупика и проверка `if найденный` — типичный приём
рекурсивного поиска: неудачные ветки просто отбрасываются.

### Пример 6. Дерево игры

Построим дерево простой игры: из числа 5 можно получить новое число
прибавлением 1 или умножением на 2; игра заканчивается, когда число
дошло до 8 или больше.

In [ ]:
def дерево_игры(число, глубина=0, предел=8):
    отступ = "  " * глубина
    if число >= предел:
        print(f"{отступ}{число} — конец")
        return 1
    print(f"{отступ}{число}")
    return дерево_игры(число + 1, глубина + 1, предел) + \
        дерево_игры(число * 2, глубина + 1, предел)


листьев_игры = дерево_игры(5)
print("\nВариантов окончания:", листьев_игры)

Каждая вершина — позиция, каждая ветка — ход. Именно так устроен
перебор в задачах ЕГЭ про выигрышные стратегии, к которым мы вернёмся
в четвёртой четверти.

## Пробуем сами

### Задача 1. Сколько рёбер

Сколько рёбер в дереве из 12 вершин?

In [ ]:
#@title 🧩 Задача 1. Рёбра дерева { display-mode: "form" }
#@markdown Впишите число
рёбер = 0 #@param {type:"integer"}

si.ответ("1", рёбер, "4fc82b26aecb47d2",
         hint="У каждой вершины, кроме корня, ровно один родитель.")

### Задача 2. Количество листьев

Функция получает дерево (словарь «вершина → список потомков»)
и корень, возвращает количество листьев.

In [ ]:
def сколько_листьев(дерево, корень):
    return ...

In [ ]:
si.check("2", сколько_листьев, [
    (({"А": ["Б", "В"], "Б": [], "В": []}, "А"), 2),
    (({"А": []}, "А"), 1),
    (({"А": ["Б"], "Б": ["В", "Г"], "В": [], "Г": []}, "А"), 2),
])

### Задача 3. Высота дерева

Функция возвращает высоту дерева. У дерева из одной вершины
высота 0.

In [ ]:
def высота_дерева(дерево, корень):
    return ...

In [ ]:
si.check("3", высота_дерева, [
    (({"А": ["Б", "В"], "Б": [], "В": []}, "А"), 1),
    (({"А": []}, "А"), 0),
    (({"А": ["Б"], "Б": ["В"], "В": ["Г"], "Г": []}, "А"), 3),
])

### Задача 4. Вершины уровня

Функция возвращает список вершин заданного уровня в порядке обхода
в ширину. Корень — уровень 0.

In [ ]:
def вершины_уровня(дерево, корень, уровень):
    return ...

In [ ]:
si.check("4", вершины_уровня, [
    (({"А": ["Б", "В"], "Б": ["Г"], "В": [], "Г": []}, "А", 1), ["Б", "В"]),
    (({"А": ["Б", "В"], "Б": ["Г"], "В": [], "Г": []}, "А", 0), ["А"]),
    (({"А": ["Б", "В"], "Б": ["Г"], "В": [], "Г": []}, "А", 5), []),
])

### Задача 5. Путь до вершины

Функция возвращает список вершин пути от корня до цели.
Если цели в дереве нет — пустой список.

In [ ]:
def путь_до(дерево, корень, цель):
    return ...

In [ ]:
si.check("5", путь_до, [
    (({"А": ["Б", "В"], "Б": ["Г"], "В": [], "Г": []}, "А", "Г"), ["А", "Б", "Г"]),
    (({"А": ["Б"], "Б": []}, "А", "А"), ["А"]),
    (({"А": ["Б"], "Б": []}, "А", "Я"), []),
])

### Задача 6. Сумма значений в поддереве

Функция получает дерево, словарь значений вершин и корень поддерева,
возвращает сумму значений всех вершин этого поддерева.

In [ ]:
def сумма_поддерева(дерево, значения, вершина):
    return ...

In [ ]:
si.check("6", сумма_поддерева, [
    (({"А": ["Б", "В"], "Б": [], "В": []}, {"А": 1, "Б": 2, "В": 3}, "А"), 6),
    (({"А": ["Б", "В"], "Б": [], "В": []}, {"А": 1, "Б": 2, "В": 3}, "Б"), 2),
    (({"А": []}, {"А": 10}, "А"), 10),
])

### Задача 7. Какой обход

Каким обходом считают размер папки с вложенными подпапками?

In [ ]:
#@title 🧩 Задача 7. Обход для размера { display-mode: "form" }
#@markdown Выберите ответ
обход = "выбери ответ" #@param ["выбери ответ", "прямой", "обратный", "в ширину"]

si.ответ("7", обход, "30fa08a2ca6926b3",
         hint="Родителя нельзя посчитать, пока не посчитаны все потомки.")

## Домашнее задание

### Домашнее задание 1. Количество вершин

Функция возвращает количество вершин в поддереве с заданным корнем.

In [ ]:
def сколько_вершин(дерево, корень):
    return ...

In [ ]:
si.check("дз1", сколько_вершин, [
    (({"А": ["Б", "В"], "Б": [], "В": []}, "А"), 3),
    (({"А": ["Б", "В"], "Б": [], "В": []}, "В"), 1),
    (({"А": ["Б"], "Б": ["В"], "В": []}, "А"), 3),
])

### Домашнее задание 2. Список всех листьев

Функция возвращает список листьев в порядке прямого обхода
(слева направо).

In [ ]:
def листья_списком(дерево, корень):
    return ...

In [ ]:
si.check("дз2", листья_списком, [
    (({"А": ["Б", "В"], "Б": ["Г", "Д"], "В": [], "Г": [], "Д": []}, "А"),
     ["Г", "Д", "В"]),
    (({"А": []}, "А"), ["А"]),
])

### Домашнее задание 3. Дерево своей папки

Постройте на бумаге дерево какой-нибудь своей папки с документами:
три-четыре уровня вложенности. Запишите его словарём, посчитайте
программой количество вершин, листьев и высоту. Потом сравните
с настоящими данными: в Colab это делает `os.walk`, а на компьютере —
свойства папки в проводнике.

---

### Любопытно

Компилятор, читая вашу программу, строит из неё дерево: корень —
вся программа, ветви — функции, дальше циклы, выражения, отдельные
операции. Выражение `2 + 3 * 4` превращается в дерево, где умножение
оказывается ниже сложения, — и именно поэтому оно выполняется
первым. Приоритет операций, о котором столько разговоров в теме
логики, физически живёт в форме этого дерева.

---

## Отчёт по уроку

Запусти ячейку ниже, когда решишь задачи. Результат уйдёт учителю автоматически.

In [ ]:
si.report()

---

[← Урок 12](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-12.ipynb) · [⬆ Все уроки](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/index.ipynb) · [🏠 Ко всем классам](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/index.ipynb) · [Урок 14 →](https://colab.research.google.com/github/LeksssGray/informatika-7-11/blob/main/klass-11/urok-14.ipynb)